# Lab 04 – SparkML: Machine Learning with Apache Spark
**Course**: CO3137 – Big Data  
**Dataset**: MovieLens Latest Small (via kagglehub → local Kafka)  
**Kafka**: `localhost:9092,localhost:9192,localhost:9292`  

| Section | Exercise | Task |
|---------|----------|------|
| Setup   | –  | SparkSession, data download, Kafka topics |
| Ex0     | 0  | Read movie data from Kafka |
| Ex1     | 1  | Binary classifier – *Will user rate ≥ 4?* |
| Ex2     | 2  | Movie clustering (K-Means + TF-IDF tags + genres) |
| Ex3     | 3  | ALS recommendation system |

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, avg, count, countDistinct, stddev,
    concat_ws, collect_list, split, concat, lit,
    from_json, to_json, struct, explode, desc, asc
)
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, FloatType, DoubleType, StringType, LongType
)
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    Tokenizer, CountVectorizer, IDF,
    VectorAssembler, StandardScaler
)
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.clustering import KMeans
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
    ClusteringEvaluator,
    RegressionEvaluator
)
from confluent_kafka.admin import AdminClient, NewTopic
import kagglehub
import time

print("All imports successful!")

All imports successful!


In [4]:
spark = (SparkSession.builder
    .appName("BigData_Lab04")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.jars.packages",
            "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.1,"
            "org.apache.kafka:kafka-clients:3.6.0,"
            "org.apache.spark:spark-streaming-kafka-0-10_2.13:4.0.1")
    .getOrCreate())

print(f"SparkSession started! Version: {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/18 22:22:58 WARN Utils: Your hostname, Kiennguyen, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/18 22:22:58 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/kiennguyen/bigdata/lab01/venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/kiennguyen/.ivy2.5.2/cache
The jars for the packages stored in: /home/kiennguyen/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
org.apache.kafka#kafka-clients added as a dependency
org.apache.spark#spark-streaming-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-84de18dd-6ba9-40c7-9f76-8ea12b267776;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.0.1 in central

SparkSession started! Version: 4.0.1


## Setup – Download Data & Push to Kafka

In [5]:
# Download MovieLens Latest Small via kagglehub
path = kagglehub.dataset_download("grouplens/movielens-latest-small")

df_ratings_raw = spark.read.csv(path + "/ratings.csv", header=True, inferSchema=True)
df_movies_raw  = spark.read.csv(path + "/movies.csv",  header=True, inferSchema=True)
df_tags_raw    = spark.read.csv(path + "/tags.csv",    header=True, inferSchema=True)

print(f"Ratings: {df_ratings_raw.count()} rows")
print(f"Movies:  {df_movies_raw.count()} rows")
print(f"Tags:    {df_tags_raw.count()} rows")
df_ratings_raw.show(2)
df_movies_raw.show(2)
df_tags_raw.show(2)

Ratings: 100836 rows
Movies:  9742 rows
Tags:    3683 rows
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
+------+-------+------+---------+
only showing top 2 rows
+-------+----------------+--------------------+
|movieId|           title|              genres|
+-------+----------------+--------------------+
|      1|Toy Story (1995)|Adventure|Animati...|
|      2|  Jumanji (1995)|Adventure|Childre...|
+-------+----------------+--------------------+
only showing top 2 rows
+------+-------+---------------+----------+
|userId|movieId|            tag| timestamp|
+------+-------+---------------+----------+
|     2|  60756|          funny|1445714994|
|     2|  60756|Highly quotable|1445714996|
+------+-------+---------------+----------+
only showing top 2 rows


In [6]:
BROKERS = "localhost:9092,localhost:9192,localhost:9292"
TOPICS  = ["Lab1_ratings", "Lab1_movies", "Lab1_tags"]

admin_client = AdminClient({"bootstrap.servers": BROKERS})

# Delete existing topics
fs = admin_client.delete_topics(TOPICS)
for topic, f in fs.items():
    try:
        f.result()
        print(f"Deleted: {topic}")
    except Exception as e:
        print(f"Could not delete {topic}: {e}")

time.sleep(2)  # wait for broker to finish deletion

# Recreate topics (3 partitions, replication factor 3)
new_topics = [NewTopic(t, num_partitions=3, replication_factor=3) for t in TOPICS]
fs = admin_client.create_topics(new_topics)
for topic, f in fs.items():
    try:
        f.result()
        print(f"Created: {topic}")
    except Exception as e:
        print(f"Error creating {topic}: {e}")

Could not delete Lab1_ratings: KafkaError{code=UNKNOWN_TOPIC_OR_PART,val=3,str="Broker: Unknown topic or partition"}
Could not delete Lab1_movies: KafkaError{code=UNKNOWN_TOPIC_OR_PART,val=3,str="Broker: Unknown topic or partition"}
Could not delete Lab1_tags: KafkaError{code=UNKNOWN_TOPIC_OR_PART,val=3,str="Broker: Unknown topic or partition"}
Created: Lab1_ratings
Created: Lab1_movies
Created: Lab1_tags


In [7]:
def push_to_kafka(df, topic):
    (df.selectExpr("to_json(struct(*)) AS value")
       .write
       .format("kafka")
       .option("kafka.bootstrap.servers", BROKERS)
       .option("topic", topic)
       .save())
    print(f"Pushed to '{topic}'")

push_to_kafka(df_ratings_raw, "Lab1_ratings")
push_to_kafka(df_movies_raw,  "Lab1_movies")
push_to_kafka(df_tags_raw,    "Lab1_tags")
print("\nAll data pushed to Kafka!")

Pushed to 'Lab1_ratings'
Pushed to 'Lab1_movies'
Pushed to 'Lab1_tags'

All data pushed to Kafka!


---
## Exercise 0 – Prepare Movie Data
Read the three topics from Kafka, define schemas, and convert data types.

In [8]:
# ── Schema definitions ────────────────────────────────────────────
ratings_schema = StructType([
    StructField("userId",    IntegerType()),
    StructField("movieId",   IntegerType()),
    StructField("rating",    DoubleType()),
    StructField("timestamp", LongType())
])

movies_schema = StructType([
    StructField("movieId", IntegerType()),
    StructField("title",   StringType()),
    StructField("genres",  StringType())
])

tags_schema = StructType([
    StructField("userId",    IntegerType()),
    StructField("movieId",   IntegerType()),
    StructField("tag",       StringType()),
    StructField("timestamp", LongType())
])

# ── Read from Kafka (batch) ───────────────────────────────────────
def read_from_kafka(topic, schema):
    return (spark.read
        .format("kafka")
        .option("kafka.bootstrap.servers", BROKERS)
        .option("subscribe", topic)
        .option("startingOffsets", "earliest")
        .load()
        .select(from_json(col("value").cast("string"), schema).alias("data"))
        .select("data.*"))

ratings_df = read_from_kafka("Lab1_ratings", ratings_schema)
movies_df  = read_from_kafka("Lab1_movies",  movies_schema)
tags_df    = read_from_kafka("Lab1_tags",    tags_schema)

print("Data read from Kafka:")
print(f"  ratings: {ratings_df.count()} rows")
print(f"  movies:  {movies_df.count()} rows")
print(f"  tags:    {tags_df.count()} rows")

Data read from Kafka:


  ratings: 100836 rows


  movies:  9742 rows
  tags:    3683 rows


In [10]:
print("=== Ratings ===")
ratings_df.printSchema()
ratings_df.show(3)

print("=== Movies ===")
movies_df.printSchema()
movies_df.show(3, truncate=False)

print("=== Tags ===")
tags_df.printSchema()
tags_df.show(3, truncate=False)

=== Ratings ===
root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: long (nullable = true)

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     4|   1283|   5.0|945078690|
|     4|   1288|   4.0|945079653|
|     4|   1291|   4.0|964538763|
+------+-------+------+---------+
only showing top 3 rows
=== Movies ===
root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)

+-------+-----------------------+-------------------------------------------+
|movieId|title                  |genres                                     |
+-------+-----------------------+-------------------------------------------+
|1      |Toy Story (1995)       |Adventure|Animation|Children|Comedy|Fantasy|
|2      |Jumanji (1995)         |Adventure|Children|Fantasy                 |
|3      |Grumpier Old Men (1995)|C

---
## Exercise 1 – Binary Classification: *Will User Rate ≥ 4?*

**Pipeline**: text (title + tags) → TF-IDF · genres → CountVectorizer · numeric aggregates  
**Outputs**: AUC, F1, confusion matrix, top-10 positive/negative feature signals

In [11]:
# ── Label: 1 if rating ≥ 4, else 0 ───────────────────────────────
labeled = ratings_df.withColumn(
    "label", when(col("rating") >= 4.0, 1.0).otherwise(0.0)
)

# ── User-level aggregates (behavior features) ─────────────────────
user_stats = (ratings_df.groupBy("userId").agg(
    avg("rating").alias("user_avg_rating"),
    count("rating").cast("double").alias("user_rating_count")
))

# ── Movie-level aggregates (popularity features) ──────────────────
movie_stats = (ratings_df.groupBy("movieId").agg(
    avg("rating").alias("movie_avg_rating"),
    count("rating").cast("double").alias("movie_rating_count")
))

# ── Tags per movie (all users' tags concatenated) ─────────────────
movie_tags_agg = (tags_df.groupBy("movieId")
    .agg(concat_ws(" ", collect_list("tag")).alias("tags_text")))

# ── Join everything into one training table ───────────────────────
ex1_data = (labeled
    .join(user_stats,  "userId",  "left")
    .join(movie_stats, "movieId", "left")
    .join(movie_tags_agg, "movieId", "left")
    .join(movies_df.select("movieId", "title", "genres"), "movieId", "left")
    .fillna({
        "tags_text": "",
        "user_avg_rating": 0.0, "user_rating_count": 0.0,
        "movie_avg_rating": 0.0, "movie_rating_count": 0.0
    })
    .withColumn("text_combined",  concat(col("title"), lit(" "), col("tags_text")))
    .withColumn("genres_tokens",  split(col("genres"), "\\|"))
)

print(f"Training dataset: {ex1_data.count()} rows")
ex1_data.select("userId", "movieId", "label",
                "user_avg_rating", "movie_avg_rating",
                "text_combined", "genres_tokens").show(3, truncate=50)

Training dataset: 100836 rows


+------+-------+-----+------------------+------------------+--------------------------------------------------+-------------------+
|userId|movieId|label|   user_avg_rating|  movie_avg_rating|                                     text_combined|      genres_tokens|
+------+-------+-----+------------------+------------------+--------------------------------------------------+-------------------+
|     4|   1283|  1.0|3.5555555555555554|4.2105263157894735|                         High Noon (1952) gunfight|   [Drama, Western]|
|     4|   1288|  1.0|3.5555555555555554| 4.015151515151516|This Is Spinal Tap (1984) heavy metal mockument...|           [Comedy]|
|     4|   1291|  1.0|3.5555555555555554| 4.046428571428572|Indiana Jones and the Last Crusade (1989) archa...|[Action, Adventure]|
+------+-------+-----+------------------+------------------+--------------------------------------------------+-------------------+
only showing top 3 rows


In [12]:
# ── Text: title + tags → TF-IDF ───────────────────────────────────
tokenizer = Tokenizer(inputCol="text_combined", outputCol="text_tokens")
cv_text   = CountVectorizer(inputCol="text_tokens", outputCol="text_vec",
                             minDF=2.0, vocabSize=500)
idf_text  = IDF(inputCol="text_vec", outputCol="text_idf")

# ── Genres: multi-hot encoding via CountVectorizer ────────────────
cv_genres = CountVectorizer(inputCol="genres_tokens", outputCol="genres_vec",
                             minDF=1.0)

# ── Assemble all features into one vector ─────────────────────────
numeric_cols = ["user_avg_rating", "user_rating_count",
                "movie_avg_rating", "movie_rating_count"]
assembler = VectorAssembler(
    inputCols=["text_idf", "genres_vec"] + numeric_cols,
    outputCol="features_raw",
    handleInvalid="skip"
)
scaler = StandardScaler(inputCol="features_raw", outputCol="features")

# ── Classifier ────────────────────────────────────────────────────
lr = LogisticRegression(
    featuresCol="features", labelCol="label",
    maxIter=20, regParam=0.01
)

# ── Build pipeline ────────────────────────────────────────────────
ex1_pipeline = Pipeline(stages=[
    tokenizer,  # 0
    cv_text,    # 1  CountVectorizer (text)
    idf_text,   # 2  IDF
    cv_genres,  # 3  CountVectorizer (genres)
    assembler,  # 4
    scaler,     # 5
    lr          # 6
])

print("Classification pipeline stages:")
for i, s in enumerate(ex1_pipeline.getStages()):
    print(f"  [{i}] {type(s).__name__}")

Classification pipeline stages:
  [0] Tokenizer
  [1] CountVectorizer
  [2] IDF
  [3] CountVectorizer
  [4] VectorAssembler
  [5] StandardScaler
  [6] LogisticRegression


In [13]:
train_ex1, test_ex1 = ex1_data.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train_ex1.count()} rows | Test: {test_ex1.count()} rows")

ex1_model   = ex1_pipeline.fit(train_ex1)
predictions = ex1_model.transform(test_ex1)

print("Sample predictions:")
predictions.select("userId", "movieId", "label",
                   "prediction", "probability").show(5)

26/05/18 22:25:11 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/05/18 22:25:11 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...


Train: 80726 rows | Test: 20110 rows


26/05/18 22:25:19 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/05/18 22:25:19 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/05/18 22:25:19 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/05/18 22:25:19 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/05/18 22:25:19 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/05/18 22:25:19 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/05/18 22:25:32 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/05/18 22:25:33 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/05/18 22:25:36 WARN InternalKafkaConsu

Sample predictions:


+------+-------+-----+----------+--------------------+
|userId|movieId|label|prediction|         probability|
+------+-------+-----+----------+--------------------+
|     7|      1|  1.0|       1.0|[0.48051251906107...|
|    19|      1|  1.0|       0.0|[0.73155405055790...|
|    27|      1|  0.0|       1.0|[0.34825972920240...|
|    43|      1|  1.0|       1.0|[0.08625694356141...|
|    57|      1|  1.0|       1.0|[0.41212281596515...|
+------+-------+-----+----------+--------------------+
only showing top 5 rows


In [14]:
# ── AUC ───────────────────────────────────────────────────────────
auc_eval = BinaryClassificationEvaluator(
    labelCol="label", metricName="areaUnderROC")
auc = auc_eval.evaluate(predictions)

# ── F1 ────────────────────────────────────────────────────────────
f1_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1")
f1 = f1_eval.evaluate(predictions)

print(f"AUC  (primary): {auc:.4f}")
print(f"F1 Score:       {f1:.4f}")

# ── Confusion matrix ──────────────────────────────────────────────
print("\nConfusion Matrix (label × prediction):")
predictions.groupBy("label", "prediction") \
    .count() \
    .orderBy("label", "prediction") \
    .show()

26/05/18 22:26:02 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/05/18 22:26:02 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/05/18 22:26:06 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...


AUC  (primary): 0.8111
F1 Score:       0.7252

Confusion Matrix (label × prediction):


26/05/18 22:26:09 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/05/18 22:26:09 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/05/18 22:26:09 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/05/18 22:26:09 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/05/18 22:26:09 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/05/18 22:26:09 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...


+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0| 7619|
|  0.0|       1.0| 2911|
|  1.0|       0.0| 2658|
|  1.0|       1.0| 7070|
+-----+----------+-----+



In [15]:
# ── Extract vocabulary from trained pipeline ──────────────────────
cv_text_model   = ex1_model.stages[1]   # CountVectorizerModel (text)
cv_genres_model = ex1_model.stages[3]   # CountVectorizerModel (genres)
lr_model        = ex1_model.stages[-1]  # LogisticRegressionModel

text_vocab   = list(cv_text_model.vocabulary)    # up to 500 text words
genres_vocab = list(cv_genres_model.vocabulary)  # ~20 genres

# Feature names follow VectorAssembler order: text_idf | genres_vec | numerics
feature_names = (
    text_vocab +
    [f"genre:{g}" for g in genres_vocab] +
    ["user_avg_rating", "user_rating_count",
     "movie_avg_rating", "movie_rating_count"]
)

coefficients = lr_model.coefficients.toArray()
n = min(len(coefficients), len(feature_names))
coef_pairs = sorted(zip(coefficients[:n], feature_names[:n]),
                    key=lambda x: x[0], reverse=True)

print("Top-10 Positive Feature Signals (→ predict rating ≥ 4):")
print(f"  {'Feature':<40}  Coefficient")
print(f"  {'-'*40}  -----------")
for coef, name in coef_pairs[:10]:
    print(f"  {name:<40}  {coef:+.4f}")

print("\nTop-10 Negative Feature Signals (→ predict rating < 4):")
print(f"  {'Feature':<40}  Coefficient")
print(f"  {'-'*40}  -----------")
for coef, name in coef_pairs[-10:]:
    print(f"  {name:<40}  {coef:+.4f}")

Top-10 Positive Feature Signals (→ predict rating ≥ 4):
  Feature                                   Coefficient
  ----------------------------------------  -----------
  movie_avg_rating                          +1.0189
  user_avg_rating                           +0.7954
  movie_rating_count                        +0.0276
  genre:Drama                               +0.0264
  (1964)                                    +0.0208
  killer                                    +0.0206
  (1999)                                    +0.0203
  v                                         +0.0203
  name                                      +0.0196
  apes                                      +0.0196

Top-10 Negative Feature Signals (→ predict rating < 4):
  Feature                                   Coefficient
  ----------------------------------------  -----------
  (2005)                                    -0.0216
  (2011)                                    -0.0236
  genre:Children                       

---
## Exercise 2 – Movie Clustering: Group Movies by Genres

**Features**: TF-IDF of per-movie tags + genre multi-hot encoding  
**Outputs**: Silhouette score for K ∈ {6, 8, 10, 12} · top-10 terms per cluster · 10 sample movies per cluster

In [16]:
# ── Tags aggregated per movie ─────────────────────────────────────
movie_tags_clust = (tags_df.groupBy("movieId")
    .agg(concat_ws(" ", collect_list("tag")).alias("tags_text")))

movies_clust = (movies_df
    .join(movie_tags_clust, "movieId", "left")
    .fillna({"tags_text": ""})
    .withColumn("genres_tokens", split(col("genres"), "\\|"))
)

print(f"Movies for clustering: {movies_clust.count()}")
movies_clust.select("title", "genres", "tags_text").show(3, truncate=60)

Movies for clustering: 9742
+------------------------+-------------------------------------------+---------------+
|                   title|                                     genres|      tags_text|
+------------------------+-------------------------------------------+---------------+
|        Toy Story (1995)|Adventure|Animation|Children|Comedy|Fantasy|pixar fun pixar|
| Grumpier Old Men (1995)|                             Comedy|Romance|      moldy old|
|Waiting to Exhale (1995)|                       Comedy|Drama|Romance|               |
+------------------------+-------------------------------------------+---------------+
only showing top 3 rows


In [17]:
# ── Feature pipeline (shared across all K) ────────────────────────
tok_c  = Tokenizer(inputCol="tags_text",   outputCol="tags_tokens")
cv_c   = CountVectorizer(inputCol="tags_tokens", outputCol="tags_vec",
                          minDF=1.0, vocabSize=500)
idf_c  = IDF(inputCol="tags_vec",  outputCol="tags_idf")
cvg_c  = CountVectorizer(inputCol="genres_tokens", outputCol="genres_vec_c",
                          minDF=1.0)
asm_c  = VectorAssembler(inputCols=["tags_idf", "genres_vec_c"],
                          outputCol="features_raw_c", handleInvalid="skip")
scl_c  = StandardScaler(inputCol="features_raw_c", outputCol="features_c")

base_clust_pipeline = Pipeline(stages=[tok_c, cv_c, idf_c, cvg_c, asm_c, scl_c])
base_clust_model    = base_clust_pipeline.fit(movies_clust)
features_clust_df   = base_clust_model.transform(movies_clust)

# ── Run KMeans for K ∈ {6, 8, 10, 12} ────────────────────────────
k_values = [6, 8, 10, 12]
sil_scores    = {}
cluster_models = {}

for k in k_values:
    km = KMeans(featuresCol="features_c", predictionCol="cluster", k=k, seed=42)
    km_model  = km.fit(features_clust_df)
    clustered = km_model.transform(features_clust_df)
    evaluator = ClusteringEvaluator(
        featuresCol="features_c", predictionCol="cluster")
    score = evaluator.evaluate(clustered)
    sil_scores[k]     = score
    cluster_models[k] = (km_model, clustered)
    print(f"K={k:2d}  Silhouette={score:.4f}")

print("\n── Silhouette Summary ──")
for k in k_values:
    bar = '█' * max(1, int(sil_scores[k] * 30))
    print(f"  K={k:2d}: {sil_scores[k]:.4f}  {bar}")

26/05/18 22:26:37 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/05/18 22:26:37 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/05/18 22:26:41 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/05/18 22:26:41 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/05/18 22:26:41 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...


K= 6  Silhouette=0.9096


26/05/18 22:26:45 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/05/18 22:26:45 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/05/18 22:26:45 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/05/18 22:26:45 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...


K= 8  Silhouette=0.8350


K=10  Silhouette=0.8364


26/05/18 22:26:53 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/05/18 22:26:53 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/05/18 22:26:53 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...


K=12  Silhouette=-0.0141

── Silhouette Summary ──
  K= 6: 0.9096  ███████████████████████████
  K= 8: 0.8350  █████████████████████████
  K=10: 0.8364  █████████████████████████
  K=12: -0.0141  █


In [18]:
# ── Detailed output for the K with the highest silhouette score ───
best_k = max(sil_scores, key=lambda k: sil_scores[k])
best_km_model, best_clustered = cluster_models[best_k]
print(f"Best K={best_k}  (Silhouette={sil_scores[best_k]:.4f})")

# Vocabulary for term mapping
tags_vocab_c   = list(base_clust_model.stages[1].vocabulary)  # cv_c vocab
genres_vocab_c = list(base_clust_model.stages[3].vocabulary)  # cvg_c vocab
full_vocab_c   = tags_vocab_c + [f"genre:{g}" for g in genres_vocab_c]

centers = best_km_model.clusterCenters()

for c in range(best_k):
    cluster_data = best_clustered.filter(col("cluster") == c)
    size = cluster_data.count()

    # Top-10 terms from cluster center (highest weight dimensions)
    center = list(centers[c])
    n_feat = min(len(center), len(full_vocab_c))
    top_idx   = sorted(range(n_feat), key=lambda i: center[i], reverse=True)[:10]
    top_terms = [full_vocab_c[i] for i in top_idx if center[i] > 0]

    print(f"\n{'='*65}")
    print(f"Cluster {c:2d}  (size={size})")
    print(f"Top terms: {top_terms}")
    print("Sample movies (top 10):")
    cluster_data.select("title", "genres").show(10, truncate=False)

Best K=6  (Silhouette=0.9096)



Cluster  0  (size=9734)
Top terms: ['', 'genre:Drama', 'genre:Comedy', 'genre:Thriller', 'genre:Action', 'genre:Romance', 'genre:Adventure', 'genre:Crime', 'genre:Horror', 'genre:Sci-Fi']
Sample movies (top 10):


26/05/18 22:27:59 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/05/18 22:27:59 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/05/18 22:27:59 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/05/18 22:27:59 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/05/18 22:27:59 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/05/18 22:27:59 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...


+------------------------------------------+--------------------------------------------+
|title                                     |genres                                      |
+------------------------------------------+--------------------------------------------+
|Awfully Big Adventure, An (1995)          |Drama                                       |
|Out of Africa (1985)                      |Drama|Romance                               |
|Children of the Corn (1984)               |Horror|Thriller                             |
|American Tail: Fievel Goes West, An (1991)|Adventure|Animation|Children|Musical|Western|
|King Kong (1933)                          |Action|Adventure|Fantasy|Horror             |
|It Came from Hollywood (1982)             |Comedy|Documentary                          |
|Chuck & Buck (2000)                       |Comedy|Drama                                |
|Hellbound: Hellraiser II (1988)           |Horror                                      |
|Dungeons 

+-------------------+---------------------------+
|title              |genres                     |
+-------------------+---------------------------+
|Pulp Fiction (1994)|Comedy|Crime|Drama|Thriller|
+-------------------+---------------------------+


Cluster  2  (size=1)
Top terms: ['hannibal', 'drama', 'gothic', 'disturbing', 'psychology', 'suspense', 'genre:Horror', 'genre:Crime', 'genre:Thriller']
Sample movies (top 10):


+--------------------------------+---------------------+
|title                           |genres               |
+--------------------------------+---------------------+
|Silence of the Lambs, The (1991)|Crime|Horror|Thriller|
+--------------------------------+---------------------+




Cluster  3  (size=1)
Top terms: ['paranoid', 'mathematics', 'insanity', 'enigmatic', 'hallucinatory', 'existentialism', 'paranoia', 'cerebral', 'tense', 'mindfuck']
Sample movies (top 10):
+---------+---------------------+
|title    |genres               |
+---------+---------------------+
|Pi (1998)|Drama|Sci-Fi|Thriller|
+---------+---------------------+




Cluster  4  (size=1)
Top terms: ['stunning', 'marvel', 'visually', 'sad', 'robert', 'emotional', 'comic', 'the', 'book', 'of']
Sample movies (top 10):
+--------------------------------------+-----------------------+
|title                                 |genres                 |
+--------------------------------------+-----------------------+
|Avengers: Infinity War - Part I (2018)|Action|Adventure|Sci-Fi|
+--------------------------------------+-----------------------+



26/05/18 22:28:08 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/05/18 22:28:08 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...



Cluster  5  (size=4)
Top terms: ['philosophy', 'thought-provoking', 'dreamlike', 'psychological', 'surreal', 'reality', 'carrey', 'jim', 'quirky', 'psychology']
Sample movies (top 10):
+--------------------------------------------+-----------------------------------------------+
|title                                       |genres                                         |
+--------------------------------------------+-----------------------------------------------+
|Inception (2010)                            |Action|Crime|Drama|Mystery|Sci-Fi|Thriller|IMAX|
|Donnie Darko (2001)                         |Drama|Mystery|Sci-Fi|Thriller                  |
|Fight Club (1999)                           |Action|Crime|Drama|Thriller                    |
|Eternal Sunshine of the Spotless Mind (2004)|Drama|Romance|Sci-Fi                           |
+--------------------------------------------+-----------------------------------------------+



---
## Exercise 3 – ALS Recommendation System

**Model**: Alternating Least Squares trained on explicit ratings  
**Outputs**: RMSE on test set · Precision@10 · top-10 films for 3 random users

In [19]:
# ALS requires Integer userId and movieId — already IntegerType from schema
# Cast rating to Float (ALS works with both Float and Double)
als_data = ratings_df.select(
    col("userId"),
    col("movieId"),
    col("rating").cast("float").alias("rating")
)

train_rec, test_rec = als_data.randomSplit([0.8, 0.2], seed=42)
print(f"ALS train: {train_rec.count()} | test: {test_rec.count()}")
als_data.show(3)

ALS train: 80774 | test: 20062
+------+-------+------+
|userId|movieId|rating|
+------+-------+------+
|     4|   1283|   5.0|
|     4|   1288|   4.0|
|     4|   1291|   4.0|
+------+-------+------+
only showing top 3 rows


In [20]:
als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    nonnegative=True,
    implicitPrefs=False,        # explicit ratings
    rank=10,                    # latent factors
    maxIter=10,
    regParam=0.1,
    coldStartStrategy="drop",   # drop NaN predictions for unseen users/items
    seed=42
)

als_model = als.fit(train_rec)
print("ALS model trained!")

ALS model trained!


In [21]:
# ── RMSE ──────────────────────────────────────────────────────────
test_predictions = als_model.transform(test_rec)
rmse_eval = RegressionEvaluator(
    metricName="rmse", labelCol="rating", predictionCol="prediction")
rmse = rmse_eval.evaluate(test_predictions.filter(col("prediction").isNotNull()))
print(f"RMSE on test set: {rmse:.4f}")

# ── Precision@10 ──────────────────────────────────────────────────
# Top-10 recommendations for every user
user_recs = als_model.recommendForAllUsers(10)

# Explode to (userId, movieId) recommendation pairs
rec_pairs = (user_recs
    .select("userId", explode("recommendations").alias("rec"))
    .select("userId", col("rec.movieId").alias("movieId")))

# "Relevant" = movies in test set rated ≥ 3.5
relevant = test_rec.filter(col("rating") >= 3.5).select("userId", "movieId")

# Count hits per user
hits = (rec_pairs
    .join(relevant, ["userId", "movieId"])
    .groupBy("userId").count()
    .withColumnRenamed("count", "hits"))

# Precision = hits / 10 — only for users with ≥1 relevant item
users_with_relevant = (relevant.groupBy("userId")
    .count().filter(col("count") >= 1).select("userId"))

rec_sizes = rec_pairs.groupBy("userId").count().withColumnRenamed("count", "num_recs")

precision_df = (rec_sizes
    .join(hits, "userId", "left")
    .fillna(0, subset=["hits"])
    .join(users_with_relevant, "userId")
    .withColumn("precision_at_10", col("hits") / col("num_recs")))

avg_p10 = precision_df.agg(avg("precision_at_10")).collect()[0][0]
n_users = precision_df.count()
print(f"Precision@10: {avg_p10:.4f}  (averaged over {n_users} users with ≥1 relevant item)")

26/05/18 22:28:56 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/05/18 22:28:56 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...


RMSE on test set: 0.8753


Precision@10: 0.0023  (averaged over 604 users with ≥1 relevant item)


In [22]:
# ── Top-10 recommendations for 3 random users ────────────────────
random_users = (train_rec.select("userId").distinct()
    .orderBy(col("userId").asc()).limit(3))
sample_uids  = [r.userId for r in random_users.collect()]

user_recs_named = (als_model
    .recommendForUserSubset(random_users, 10)
    .select("userId", explode("recommendations").alias("rec"))
    .select("userId",
            col("rec.movieId").alias("movieId"),
            col("rec.rating").alias("pred_rating"))
    .join(movies_df.select("movieId", "title", "genres"), "movieId", "left"))

print("Top-10 Film Recommendations for 3 Random Users\n")
for uid in sample_uids:
    print(f"── User {uid} ──")
    (user_recs_named
        .filter(col("userId") == uid)
        .select("title", "genres", "pred_rating")
        .orderBy(desc("pred_rating"))
        .show(10, truncate=False))

Top-10 Film Recommendations for 3 Random Users

── User 1 ──
+-------------------------------------------------------------------------------------------------+-------------------------------+-----------+
|title                                                                                            |genres                         |pred_rating|
+-------------------------------------------------------------------------------------------------+-------------------------------+-----------+
|Saving Face (2004)                                                                               |Comedy|Drama|Romance           |5.9873533  |
|Seve (2014)                                                                                      |Documentary|Drama              |5.8054276  |
|The Big Bus (1976)                                                                               |Action|Comedy                  |5.8054276  |
|Victory (a.k.a. Escape to Victory) (1981)                                 

26/05/18 22:29:04 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/05/18 22:29:04 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/05/18 22:29:04 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...


+----------------------------------------------------+--------------------+-----------+
|title                                               |genres              |pred_rating|
+----------------------------------------------------+--------------------+-----------+
|Reign Over Me (2007)                                |Drama               |4.986836   |
|Saving Face (2004)                                  |Comedy|Drama|Romance|4.9398     |
|The Jinx: The Life and Deaths of Robert Durst (2015)|Documentary         |4.8774767  |
|Love and Death (1975)                               |Comedy              |4.8527107  |
|Half Nelson (2006)                                  |Drama               |4.722693   |
|De platte jungle (1978)                             |Documentary         |4.7207584  |
|Blue Planet II (2017)                               |Documentary         |4.7207584  |
|Watermark (2014)                                    |Documentary         |4.7207584  |
|Connections (1978)             

+------------------------------------------------------------------------+-------------------------------------+-----------+
|title                                                                   |genres                               |pred_rating|
+------------------------------------------------------------------------+-------------------------------------+-----------+
|Phantasm II (1988)                                                      |Action|Fantasy|Horror|Sci-Fi|Thriller|5.135066   |
|Death Race 2000 (1975)                                                  |Action|Sci-Fi                        |4.948452   |
|Alien Contamination (1980)                                              |Action|Horror|Sci-Fi                 |4.948452   |
|Android (1982)                                                          |Sci-Fi                               |4.948452   |
|Galaxy of Terror (Quest) (1981)                                         |Action|Horror|Mystery|Sci-Fi         |4.948452   |
